# ```Feature Engineering of IPO Data```

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
ipo = pd.read_csv(r"C:\Users\Rano's PC\Machine\InsightForge\InsightForge\Avoiding The Hype Trap\_datasets\_cleaned\ipo.csv")


price = pd.read_csv(r"C:\Users\Rano's PC\Machine\InsightForge\InsightForge\Avoiding The Hype Trap\_datasets\_cleaned\prices.csv")

In [3]:
ipo.head()

,company,listing_date,final_price,price_change,retail_subscription
0,Prince Pipes & Fittings Ltd.,2019-12-30,178,-6.24,1.63
1,Ujjivan Small Finance Bank Ltd.,2019-12-12,37,51.22,45.29
2,CSB Bank Ltd.,2019-12-04,195,53.87,40.98
3,Vishwaraj Sugar Industries Ltd.,2019-10-15,60,0.58,0.59
4,Indian Railway Catering & Tourism Corp.Ltd.,2019-10-14,320,127.42,13.28


In [4]:
price.head()

,company,date,open,high,low,close,volume
0,Prince Pipes & Fittings Ltd.,2019-12-30,160.00,177.9,152.50,166.90,21159947
1,Prince Pipes & Fittings Ltd.,2019-12-31,162.30,167.4,150.60,152.45,5939748
2,Prince Pipes & Fittings Ltd.,2020-01-01,153.05,159.5,152.40,155.85,3114230
3,Prince Pipes & Fittings Ltd.,2020-01-02,156.05,158.7,154.00,155.70,1021302
4,Prince Pipes & Fittings Ltd.,2020-01-03,156.00,156.0,151.25,151.90,623408


In [6]:
price['date']=pd.to_datetime(price['date'])
ipo['listing_date']= pd.to_datetime(ipo['listing_date'])

In [7]:
from datetime import datetime

ipo['days_since_listing'] = (pd.Timestamp.today().normalize() - ipo['listing_date']).dt.days

In [8]:
ipo.head()

,company,listing_date,final_price,price_change,retail_subscription,days_since_listing
0,Prince Pipes & Fittings Ltd.,2019-12-30,178,-6.24,1.63,2390
1,Ujjivan Small Finance Bank Ltd.,2019-12-12,37,51.22,45.29,2408
2,CSB Bank Ltd.,2019-12-04,195,53.87,40.98,2416
3,Vishwaraj Sugar Industries Ltd.,2019-10-15,60,0.58,0.59,2466
4,Indian Railway Catering & Tourism Corp.Ltd.,2019-10-14,320,127.42,13.28,2467


In [9]:

# make sure dates are sorted for each company
price = price.sort_values(['company', 'date'])

def get_price_at(group, listing_date, days):
    target_date = listing_date + pd.Timedelta(days=days)
    window = group[group['date'] <= target_date]
    if window.empty:
        return np.nan
    return window.iloc[-1]['close']

def compute_features(row):
    company = row['company']
    listing_date = row['listing_date']
    final_price = row['final_price']
    
    group = price[price['company'] == company]
    if group.empty:
        return pd.Series([np.nan]*6)
    
    # fixed-window returns
    r_1w = get_price_at(group, listing_date, 7)
    r_1m = get_price_at(group, listing_date, 30)
    r_3m = get_price_at(group, listing_date, 90)
    r_6m = get_price_at(group, listing_date, 180)
    r_1y = get_price_at(group, listing_date, 365)
    
    ret_1w = (r_1w - final_price) / final_price * 100 if pd.notna(r_1w) else np.nan
    ret_1m = (r_1m - final_price) / final_price * 100 if pd.notna(r_1m) else np.nan
    ret_3m = (r_3m - final_price) / final_price * 100 if pd.notna(r_3m) else np.nan
    ret_6m = (r_6m - final_price) / final_price * 100 if pd.notna(r_6m) else np.nan
    ret_1y = (r_1y - final_price) / final_price * 100 if pd.notna(r_1y) else np.nan
    
    # volatility (std of daily returns, first 90 days)
    early = group[group['date'] <= listing_date + pd.Timedelta(days=90)]
    daily_returns = early['close'].pct_change()
    volatility = daily_returns.std() * 100
    
    return pd.Series([ret_1w, ret_1m, ret_3m, ret_6m, ret_1y, volatility])

ipo[['ret_1w', 'ret_1m', 'ret_3m', 'ret_6m', 'ret_1y', 'volatility']] = ipo.apply(compute_features, axis=1)

ipo.head()

,company,listing_date,final_price,price_change,retail_subscription,days_since_listing,ret_1w,ret_1m,ret_3m,ret_6m,ret_1y,volatility
0,Prince Pipes & Fittings Ltd.,2019-12-30,178,-6.24,1.63,2390,-15.617978,-0.084270,-44.325843,-35.000000,58.511236,3.872215
1,Ujjivan Small Finance Bank Ltd.,2019-12-12,37,51.22,45.29,2408,59.729730,41.081081,11.621622,-18.783784,5.000000,3.128991
2,CSB Bank Ltd.,2019-12-04,195,53.87,40.98,2416,23.435897,6.692308,-10.435897,-34.820513,15.282051,2.789432
3,Vishwaraj Sugar Industries Ltd.,2019-10-15,60,0.58,0.59,2466,-79.850000,-77.866667,-69.950000,-78.333333,-63.283333,4.910541
4,Indian Railway Catering & Tourism Corp.Ltd.,2019-10-14,320,127.42,13.28,2467,-51.303125,-41.746875,-42.671875,-21.631250,-16.468750,2.356748


In [10]:
def compute_max_drawdown(row):
    company = row['company']
    listing_date = row['listing_date']
    
    group = price[(price['company'] == company) & 
                   (price['date'] <= listing_date + pd.Timedelta(days=365))]
    
    if group.empty:
        return np.nan
    
    prices_series = group['close']
    running_max = prices_series.cummax()
    drawdown = (prices_series - running_max) / running_max * 100
    return drawdown.min()  # most negative value = worst drop

ipo['max_drawdown'] = ipo.apply(compute_max_drawdown, axis=1)

# Day 1 gain bucket
def bucket_day1(x):
    if pd.isna(x):
        return np.nan
    elif x < 10:
        return 'low'
    elif x < 50:
        return 'medium'
    else:
        return 'high'

ipo['gain_bucket'] = ipo['price_change'].apply(bucket_day1)

ipo.head()

,company,listing_date,final_price,price_change,retail_subscription,days_since_listing,ret_1w,ret_1m,ret_3m,ret_6m,ret_1y,volatility,max_drawdown,gain_bucket
0,Prince Pipes & Fittings Ltd.,2019-12-30,178,-6.24,1.63,2390,-15.617978,-0.084270,-44.325843,-35.000000,58.511236,3.872215,-60.220704,low
1,Ujjivan Small Finance Bank Ltd.,2019-12-12,37,51.22,45.29,2408,59.729730,41.081081,11.621622,-18.783784,5.000000,3.128991,-59.221658,high
2,CSB Bank Ltd.,2019-12-04,195,53.87,40.98,2416,23.435897,6.692308,-10.435897,-34.820513,15.282051,2.789432,-66.772205,high
3,Vishwaraj Sugar Industries Ltd.,2019-10-15,60,0.58,0.59,2466,-79.850000,-77.866667,-69.950000,-78.333333,-63.283333,4.910541,-37.526205,low
4,Indian Railway Catering & Tourism Corp.Ltd.,2019-10-14,320,127.42,13.28,2467,-51.303125,-41.746875,-42.671875,-21.631250,-16.468750,2.356748,-58.226389,high


In [13]:
pd.set_option('display.max_columns', 20)

In [14]:
ipo.head()

,company,listing_date,final_price,price_change,retail_subscription,days_since_listing,ret_1w,ret_1m,ret_3m,ret_6m,ret_1y,volatility,max_drawdown,gain_bucket
0,Prince Pipes & Fittings Ltd.,2019-12-30,178,-6.24,1.63,2390,-15.617978,-0.084270,-44.325843,-35.000000,58.511236,3.872215,-60.220704,low
1,Ujjivan Small Finance Bank Ltd.,2019-12-12,37,51.22,45.29,2408,59.729730,41.081081,11.621622,-18.783784,5.000000,3.128991,-59.221658,high
2,CSB Bank Ltd.,2019-12-04,195,53.87,40.98,2416,23.435897,6.692308,-10.435897,-34.820513,15.282051,2.789432,-66.772205,high
3,Vishwaraj Sugar Industries Ltd.,2019-10-15,60,0.58,0.59,2466,-79.850000,-77.866667,-69.950000,-78.333333,-63.283333,4.910541,-37.526205,low
4,Indian Railway Catering & Tourism Corp.Ltd.,2019-10-14,320,127.42,13.28,2467,-51.303125,-41.746875,-42.671875,-21.631250,-16.468750,2.356748,-58.226389,high


In [15]:
price.info()

<class 'pandas.core.frame.DataFrame'>
Index: 266370 entries, 208200 to 193911
Data columns (total 7 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   company  266370 non-null  object        
 1   date     266370 non-null  datetime64[ns]
 2   open     266370 non-null  float64       
 3   high     266370 non-null  float64       
 4   low      266370 non-null  float64       
 5   close    265992 non-null  float64       
 6   volume   266370 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 16.3+ MB


In [16]:
ipo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   company               379 non-null    object        
 1   listing_date          379 non-null    datetime64[ns]
 2   final_price           379 non-null    int64         
 3   price_change          379 non-null    float64       
 4   retail_subscription   379 non-null    float64       
 5   days_since_listing    379 non-null    int64         
 6   ret_1w                377 non-null    float64       
 7   ret_1m                378 non-null    float64       
 8   ret_3m                378 non-null    float64       
 9   ret_6m                378 non-null    float64       
 10  ret_1y                302 non-null    float64       
 11  volatility            378 non-null    float64       
 12  max_drawdown          378 non-null    float64       
 13  gain_bucket         

In [17]:
price.to_csv(r"C:\Users\Rano's PC\Machine\InsightForge\InsightForge\Avoiding The Hype Trap\_datasets\_engineered_data\prices.csv",index = False)


ipo.to_csv(r"C:\Users\Rano's PC\Machine\InsightForge\InsightForge\Avoiding The Hype Trap\_datasets\_engineered_data\ipo.csv",index=False)